# UQRoute-TC — Qwen 2.5 1.5B development output-format audit

**Scope:** One clean development case each from APIBank, RotBench, ToolAlpaca, and ToolEyes. BFCL token alignment was evaluated on 20 saved pilot responses. Held-out groups are excluded.

**Recorded evidence:** Raw output, parsed calls, chosen-token log-probabilities, parser status, and token-alignment status. Records and audit summary are saved in Drive.


## 1. Data and code revisions

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys

drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/UQRoute-TC')
root.mkdir(parents=True, exist_ok=True)
benchmark = Path('/content/robustbench-tc-release')
benchmark_revision = 'd5d03180de41eb30a6c796d04a9bfbd9dad85c1d'
if not benchmark.exists():
    subprocess.run(['git', 'clone', 'https://github.com/WillChow66/robustbench-tc-release.git', str(benchmark)], check=True)
subprocess.run(['git', '-C', str(benchmark), 'checkout', '--detach', benchmark_revision], check=True)
assert subprocess.check_output(['git', '-C', str(benchmark), 'rev-parse', 'HEAD'], text=True).strip() == benchmark_revision
data_dir = benchmark / 'hf_data/datasets/api_eval'
assert not subprocess.check_output(['git', '-C', str(benchmark), 'status', '--porcelain', '--', str(data_dir)], text=True).strip(), 'Benchmark data has local changes: use a clean clone'

project = Path('/content/uqroute-tc-format-audit')
code_revision = '56fb7874db094723f61e37c63cee9900dbd629e3'
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/RavindraSSK/uqroute-tc.git', str(project)], check=True)
subprocess.run(['git', '-C', str(project), 'fetch', 'origin', 'feat/task2-canonical-parser'], check=True)
subprocess.run(['git', '-C', str(project), 'checkout', '--detach', code_revision], check=True)
assert subprocess.check_output(['git', '-C', str(project), 'rev-parse', 'HEAD'], text=True).strip() == code_revision
print('Drive:', root)
print('Benchmark:', benchmark_revision)
print('Project code:', code_revision)


## 2. Inference environment

In [ ]:
import importlib.metadata
import json
import time
import urllib.request

try:
    installed_vllm = importlib.metadata.version('vllm')
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None
if installed_vllm != '0.30.0':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.30.0', 'openai>=1.50,<3', '-e', str(project)], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.50,<3', '-e', str(project)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'], check=True)
import torch
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0), 'Select an L4 GPU runtime'
assert importlib.metadata.version('vllm') == '0.30.0'

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
model_revision = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
endpoint = 'http://127.0.0.1:8000/v1'
server_log = Path('/content/uqroute_qwen15b_format_audit_server.log')
def served_models():
    try:
        with urllib.request.urlopen(f'{endpoint}/models', timeout=3) as response:
            return [item['id'] for item in json.load(response)['data']]
    except Exception:
        return None

active = served_models()
if active is not None and model_id not in active:
    raise RuntimeError(f'Port 8000 is serving another model: {active}')
if active and model_id in active:
    print('Server ready:', model_id)
else:
    processes = subprocess.check_output(['ps', '-eo', 'args='], text=True)
    if f'vllm serve {model_id}' in processes:
        print('Existing server is starting; waiting.')
        server = None
    else:
        with server_log.open('w') as log:
            server = subprocess.Popen(['vllm', 'serve', model_id, '--revision', model_revision,
                '--tokenizer-revision', model_revision, '--dtype', 'bfloat16',
                '--max-model-len', '8192', '--gpu-memory-utilization', '0.88',
                '--port', '8000'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        print('Started server PID:', server.pid)
    for attempt in range(120):
        active = served_models()
        if active and model_id in active:
            print('Server ready:', model_id)
            break
        if server is not None and server.poll() is not None:
            print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
            raise RuntimeError('Server exited during startup')
        time.sleep(5)
    else:
        print('\n'.join(server_log.read_text(errors='replace').splitlines()[-35:]))
        raise TimeoutError('Server was not ready within ten minutes')
print('GPU:', torch.cuda.get_device_name(0), '| vLLM:', importlib.metadata.version('vllm'))


## 3. Development case generation

In [ ]:
import hashlib
from openai import OpenAI
sys.path.insert(0, str(benchmark / 'scripts'))
sys.path.insert(0, str(project / 'src'))
from run_eval import build_messages, load_samples, parse_tool_calls
from uqroute_tc.inference.pilot import run_cases

split_path = project / 'docs/splits/group_split_v1.json'
split = json.loads(split_path.read_text())
assert split['benchmark_revision'] == benchmark_revision
wanted = ('apibank', 'rotbench', 'toolalpaca', 'tooleyes')
selected = {}
for sample in load_samples(data_dir / 'clean.jsonl'):
    row_id = str(sample['id'])
    assignment = split['assignments'].get(row_id)
    if assignment == {'benchmark': sample['benchmark'], 'population': 'primary', 'split': 'development'}:
        if sample['benchmark'] in wanted:
            selected.setdefault(sample['benchmark'], sample)
assert set(selected) == set(wanted), f'Missing development benchmark(s): {set(wanted) - set(selected)}'
samples = [selected[name] for name in wanted]
print('Selected development cases:', [(sample['benchmark'], sample['id']) for sample in samples])

output_dir = root / 'pilots/qwen25-15b-format-dev-four-v1'
config = {'benchmark_revision': benchmark_revision,
          'split_sha256': hashlib.sha256(split_path.read_bytes()).hexdigest(),
          'code_revision': code_revision, 'model_id': model_id,
          'model_revision': model_revision, 'tokenizer_revision': model_revision,
          'endpoint': endpoint, 'temperature': 0.001, 'max_tokens': 1024,
          'top_logprobs': 5, 'engine_version': '0.30.0', 'dtype': 'bfloat16',
          'max_model_len': 8192, 'mode': 'single_prompt_tool_calling',
          'selection': 'first clean development row per benchmark in pinned file order',
          'case_ids': [str(sample['id']) for sample in samples]}
assert model_id in (served_models() or []), 'Qwen 1.5B server unavailable'
client = OpenAI(base_url=endpoint, api_key='EMPTY', timeout=60, max_retries=0)
processed, resumed, failed = run_cases(samples, client, build_messages, parse_tool_calls, output_dir, config)
print(f'Four-case collection: processed={processed} resumed={resumed} failed={failed}')
print('Records saved:', output_dir)


## 4. Parser and token-alignment audit

In [ ]:
from uqroute_tc.parsing.canonical import parse_prediction
from uqroute_tc.uncertainty.alignment import meaningful_token_score
from uqroute_tc.inference.pilot import _record_path

entries = []
for sample in samples:
    row = json.loads(_record_path(output_dir, str(sample['id'])).read_text())
    raw = row['prediction']['raw_output']
    parsed = parse_prediction(raw, sample['benchmark'], parse_tool_calls)
    response = row['uqroute']['response']
    choice = response['choices'][0] if response else {}
    tokens = (choice.get('logprobs') or {}).get('content') or []
    entry = {'benchmark': sample['benchmark'], 'case_id': sample['id'],
             'record_status': row['uqroute']['status'],
             'parser_status': parsed.status, 'parser_error': parsed.error,
             'parsed_calls_match_record': list(parsed.calls) == row['prediction']['tool_calls'],
             'parsed_call_count': len(parsed.calls),
             'token_evidence_valid': row['uqroute']['evidence']['valid'],
             'finish_reason': row['uqroute']['finish_reason'],
             'elapsed_seconds': row['uqroute']['elapsed_seconds'],
             'raw_output': raw}
    try:
        score = meaningful_token_score(raw, list(parsed.calls), tokens, entry['finish_reason'])
        entry['meaningful_token_status'] = 'aligned'
        entry['meaningful_token_count'] = len(score.selected_token_indices)
        entry['boundary_crossing_token_count'] = len(score.boundary_crossing_token_indices)
    except (ValueError, TypeError) as exc:
        entry['meaningful_token_status'] = 'unsupported_or_invalid'
        entry['meaningful_token_reason'] = str(exc)
    entries.append(entry)
    print(sample['benchmark'], '|', sample['id'], '|', entry['record_status'],
          '| parser:', parsed.status, '| calls:', len(parsed.calls),
          '| meaningful:', entry['meaningful_token_status'])
    print('  raw:', repr(raw[:280]))
    if entry.get('meaningful_token_reason'):
        print('  reason:', entry['meaningful_token_reason'])

audit_path = output_dir / 'format_audit.json'
audit_path.write_text(json.dumps({'manifest': config, 'entries': entries}, ensure_ascii=False, indent=2) + '\n')
print('Saved:', audit_path)
print('Audited records:', len(entries))


## 5. Official per-case scores

In [ ]:
scores_path = output_dir / 'official_scores.json'
per_case_path = output_dir / 'official_per_case.jsonl'
subprocess.run([sys.executable, str(benchmark / 'leaderboard/scoring/eval_results.py'),
                '--predictions', str(output_dir),
                '--files', 'pilot.predictions.jsonl',
                '--output', str(scores_path),
                '--per-sample-output', str(per_case_path)], check=True)
scores = [json.loads(line) for line in per_case_path.read_text().splitlines() if line.strip()]
assert len(scores) == len(samples) and {row['id'] for row in scores} == {str(sample['id']) for sample in samples}
for row in scores:
    print(row['benchmark'], '| correct:', row['score']['correct'], '| score:', row['score']['score'])
print('Per-case scores:', per_case_path)
